# Convert BDFR JSON to CSV for Sentiment Analysis
This notebook converts all JSON files from the BDFR archive into a single CSV file ready for sentiment analysis.

In [39]:
import json
import csv
import os
import re
from pathlib import Path
import pandas as pd
from textblob import TextBlob
import emoji
import string
from datetime import datetime

# Folder del raw data
json_folder = './jsons'

In [50]:
def extract_text_features(text):
    if text is None or pd.isna(text):
        return {
            'text_length': 0,
            'word_count': 0,
            'question_mark_count': 0,
            'exclamation_mark_count': 0,
            'sentiment_score': None,
            'sentiment_subjectivity': None,
            'avg_word_length': 0,
            'unique_word_ratio': 0,
            'uppercase_ratio': 0,
            'punctuation_count': 0,
            'has_url': 0,
            'has_mention': 0,
            'has_emoji': 0,
            'caps_words_count': 0,
            'has_suggested_subreddit': 0,
            'suggested_subreddit': []
        }
    text = str(text)

    # Guardar si tiene emojis ANTES de modificar
    has_emoji_val = 1 if emoji.emoji_list(text) else 0
    
    # Luego demojizar
    text = emoji.demojize(text)
    
    # Basic text metrics
    text_length = len(text)
    words = text.split()
    word_count = len(words)
    # Punctuation
    question_mark_count = text.count('?')
    exclamation_mark_count = text.count('!')
    punctuation_count = len([c for c in text if c in string.punctuation])
    # Sentiment analysis - CORREGIDO
    try:
        blob = TextBlob(text)
        sentiment_score, sentiment_subjectivity = blob.sentiment
    except Exception:
        sentiment_score, sentiment_subjectivity = None, None
    # Word complexity
    avg_word_length = sum(len(word) for word in words) / word_count if word_count > 0 else 0
    unique_words = len(set(word.lower() for word in words))
    unique_word_ratio = unique_words / word_count if word_count > 0 else 0
    # Case analysis
    uppercase_chars = sum(1 for c in text if c.isupper())
    uppercase_ratio = uppercase_chars / len(text) if len(text) > 0 else 0
    caps_words = sum(1 for word in words if word.isupper() and len(word) > 1)
    # Special patterns
    has_url = 1 if re.search(r'http[s]?://|www\.', text) else 0
    has_mention = 1 if re.search(r'/u/|u/|@', text) else 0
    suggested_subreddit = re.findall(r'\br/([a-zA-Z0-9_]+)', text, re.IGNORECASE)
    has_suggested_subreddit = 1 if suggested_subreddit else 0
    return {
        'text_length': text_length,
        'word_count': word_count,
        'question_mark_count': question_mark_count,
        'exclamation_mark_count': exclamation_mark_count,
        'sentiment_score': sentiment_score,
        'sentiment_subjectivity': sentiment_subjectivity,
        'avg_word_length': avg_word_length,
        'unique_word_ratio': unique_word_ratio,
        'uppercase_ratio': uppercase_ratio,
        'punctuation_count': punctuation_count,
        'has_url': has_url,
        'has_mention': has_mention,
        'has_suggested_subreddit': has_suggested_subreddit,
        'suggested_subreddit': suggested_subreddit,
        'has_emoji': has_emoji_val,
        'caps_words_count': caps_words,
    }
 
def extract_submissions_and_comments(json_file):
    """Extract submission data and all comments from a JSON file"""
    data = []
    
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            post = json.load(f)
        
        # Extract text features from submission
        submission_text = post.get('selftext')
        submission_features = extract_text_features(submission_text)
        
        # Add submission itself
        submission = {
            'type': 'submission',
            'title': post.get('title'),
            'text': submission_text,
            'score': post.get('score'),
            'upvote_ratio': post.get('upvote_ratio'),
            'num_comments': post.get('num_comments'),
            'created_utc': post.get('created_utc'),
            **submission_features  # Unpack all text features
        }
        data.append(submission)
        
        # Add all comments recursively
        def extract_comments(comments):
            for comment in comments:
                comment_text = comment.get('body')
                comment_features = extract_text_features(comment_text)
                
                comment_data = {
                    'type': 'comment',
                    'title': None,
                    'text': comment_text,
                    'score': comment.get('score'),
                    'upvote_ratio': None,
                    'num_comments': None,
                    'created_utc': comment.get('created_utc'),
                    **comment_features  # Unpack all text features
                }
                data.append(comment_data)
                
                # Recursively add nested replies
                if comment.get('replies'):
                    extract_comments(comment['replies'])
        
        if post.get('comments'):
            extract_comments(post['comments'])
        
        return data
    
    except Exception as e:
        print(f"Error processing {json_file}: {e}")
        return []

In [51]:
# Find all JSON files
json_files = list(Path(json_folder).glob("**/*.json"))
print(f"Found {len(json_files)} JSON files")

# Show first few files
if json_files:
    print("\nFirst few files:")
    for f in json_files[:5]:
        print(f"  - {f.name}")

Found 881 JSON files

First few files:
  - AuthoringInProgress_It's easy to get bogged down in the darker parts of being Trans, so, for a change of pace, what are your bestfunniest transition stories?_oveq36.json
  - OverexposedPotato_Transphobia got to a point that cis women with rougher features are being called “guys” and pressured into “proving they are women”_ljp2pi.json
  - SummaryExecutions_I wasn't prepared for the looks of pity._og8sb4.json
  - SingingForRin_Discovering you're trans: anyone else hit calm but also doubt?_1waiqpd.json
  - ImaginaryRoom055_How do autistic people differentiate gender dysphoria from generalised malaise?_1wbm498.json


In [56]:
# Process all JSON files
all_rows = []

for i, json_file in enumerate(json_files, 1):
    if i % 10 == 0 or i == 1:
        print(f"Processing {i}/{len(json_files)}: {json_file.name}")
    rows = extract_submissions_and_comments(str(json_file))
    all_rows.extend(rows)

print(f"\n✓ Extracted {len(all_rows)} total rows")

Processing 1/881: AuthoringInProgress_It's easy to get bogged down in the darker parts of being Trans, so, for a change of pace, what are your bestfunniest transition stories?_oveq36.json
Processing 10/881: MondoMania9_Best way to handle my thoughtsfeel better_1wc5ct2.json
Processing 20/881: Lielushhh_How many milligrams of Estrofem do you take orally?_1wd66tn.json
Processing 30/881: C0mradekitty_Would it be weird to do this with my name?_1wc23o5.json
Processing 40/881: Acceptable-Lake-6038_Oral sex tips for a cis woman dating a trans man_1waithq.json
Processing 50/881: Blu_b_rry05_This is something I’ve always wondered about_1wa5g2h.json
Processing 60/881: SophieCalle_A very, very scary result from a question given to a Trump group, showing motivation behind the memo_9qo9ra.json
Processing 70/881: No-Individual-5527_Should I try to resume partial transitioning?_1wbinn1.json
Processing 80/881: Petrychorr_What are some downsides to passing regularly that don't get talked about?_1wdh446.

In [57]:
# Convert to DataFrame
df = pd.DataFrame(all_rows)

# Show stats
print(f"Total rows: {len(df)}")
print(f"Submissions: {len(df[df['type'] == 'submission'])}")
print(f"Comments: {len(df[df['type'] == 'comment'])}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Total rows: 66659
Submissions: 881
Comments: 65778

Columns: ['type', 'title', 'text', 'score', 'upvote_ratio', 'num_comments', 'created_utc', 'text_length', 'word_count', 'question_mark_count', 'exclamation_mark_count', 'sentiment_score', 'sentiment_subjectivity', 'avg_word_length', 'unique_word_ratio', 'uppercase_ratio', 'punctuation_count', 'has_url', 'has_mention', 'has_suggested_subreddit', 'suggested_subreddit', 'has_emoji', 'caps_words_count']

First few rows:


,type,title,text,score,upvote_ratio,num_comments,created_utc,text_length,word_count,question_mark_count,...,avg_word_length,unique_word_ratio,uppercase_ratio,punctuation_count,has_url,has_mention,has_suggested_subreddit,suggested_subreddit,has_emoji,caps_words_count
0,submission,It's easy to get bogged down in the darker par...,I mean any kind of stories regarding your tran...,1092,0.99,300.0,1.627764e+09,507,88,1,...,4.750000,0.875000,0.023669,25,0,0,0,[],0,1
1,comment,NaN,"I told my mother I am trans, and she said she ...",576,NaN,NaN,1.627770e+09,171,37,0,...,3.648649,0.810811,0.023392,3,0,0,0,[],0,0
2,comment,NaN,"Either your mother is a witch, or you are.",93,NaN,NaN,1.627800e+09,42,9,0,...,3.777778,1.000000,0.023810,2,0,0,0,[],0,0
3,comment,NaN,Why not both?,64,NaN,NaN,1.627800e+09,13,3,1,...,3.666667,1.000000,0.076923,1,0,0,0,[],0,0
4,comment,NaN,I've not talked about this much but as a child...,25,NaN,NaN,1.627817e+09,679,126,0,...,4.380952,0.746032,0.016200,17,0,0,0,[],0,0


In [58]:
# Save to CSV
output_file = 'asktransgender_dataset.csv'
df.to_csv(output_file, index=False, encoding='utf-8')
print(f"✓ Saved to {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")

✓ Saved to asktransgender_dataset.csv
File size: 27.66 MB


## Optional: Create comments-only CSV for sentiment analysis

In [59]:
# If you only want comments (recommended for sentiment analysis)
comments_df = df[df['type'] == 'comment'].copy()
comments_df = comments_df[comments_df['text'].notna()]  # Remove deleted comments

print(f"Comments (excluding deleted): {len(comments_df)}")
print(f"\nSample comment:")
print(comments_df['text'].iloc[0][:200] if len(comments_df) > 0 else "No comments")

comments_output = 'asktransgender_comments_only.csv'
comments_df.to_csv(comments_output, index=False, encoding='utf-8')
print(f"✓ Saved comments to {comments_output}")
print(f"File size: {os.path.getsize(comments_output) / 1024 / 1024:.2f} MB")

Comments (excluding deleted): 65778

Sample comment:
I told my mother I am trans, and she said she had a dream where I showed up at her door as a woman a month prior. Now she thinks we have a psychic connection or something.
✓ Saved comments to asktransgender_comments_only.csv
File size: 26.43 MB


In [60]:
# If you only want submissions 
submissions_df = df[df['type'] == 'submission'].copy()
submissions_df = submissions_df[submissions_df['text'].notna()]  # Remove deleted comments

print(f"submissions (excluding deleted): {len(submissions_df)}")
print(f"\nSample submissions:")
print(submissions_df['text'].iloc[0][:200] if len(submissions_df) > 0 else "No submissions")

submissions_output = 'asktransgender_submissions_only.csv'
submissions_df.to_csv(submissions_output, index=False, encoding='utf-8')
print(f"✓ Saved submissions to {submissions_output}")
print(f"File size: {os.path.getsize(submissions_output) / 1024 / 1024:.2f} MB")

submissions (excluding deleted): 881

Sample submissions:
I mean any kind of stories regarding your transition, be that a case of mistaken identity straight out of a soap opera, egg moments so gosh darn eggy you look back and wonder "How the *hell* didn't I 
✓ Saved submissions to asktransgender_submissions_only.csv
File size: 1.23 MB


## Informacion del datset

## Data Exploration

In [61]:
# Average score by type
print("Average score by type:")
print(df.groupby('type')['score'].describe())

Average score by type:
              count        mean         std    min  25%   50%     75%     max
type                                                                         
comment     65778.0   18.402384   65.368183 -255.0  1.0   3.0    12.0  1930.0
submission    881.0  712.544835  852.564439    0.0  2.0  44.0  1276.0  4915.0


In [62]:
# Date range
df['date'] = pd.to_datetime(df['created_utc'], unit='s')
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nPosts per month:")
df.set_index('date').resample('MS').size().tail(10)

Date range: 2017-06-15 07:10:25 to 2026-09-11 22:47:22

Posts per month:


date
2025-12-01     255
2026-01-01     315
2026-02-01     679
2026-03-01      28
2026-04-01       0
2026-05-01       0
2026-06-01     517
2026-07-01     618
2026-08-01      15
2026-09-01    5472
Freq: MS, dtype: int64

In [63]:
df.head()


,type,title,text,score,upvote_ratio,num_comments,created_utc,text_length,word_count,question_mark_count,...,unique_word_ratio,uppercase_ratio,punctuation_count,has_url,has_mention,has_suggested_subreddit,suggested_subreddit,has_emoji,caps_words_count,date
0,submission,It's easy to get bogged down in the darker par...,I mean any kind of stories regarding your tran...,1092,0.99,300.0,1.627764e+09,507,88,1,...,0.875000,0.023669,25,0,0,0,[],0,1,2021-07-31 20:46:42
1,comment,NaN,"I told my mother I am trans, and she said she ...",576,NaN,NaN,1.627770e+09,171,37,0,...,0.810811,0.023392,3,0,0,0,[],0,0,2021-07-31 22:24:34
2,comment,NaN,"Either your mother is a witch, or you are.",93,NaN,NaN,1.627800e+09,42,9,0,...,1.000000,0.023810,2,0,0,0,[],0,0,2021-08-01 06:40:00
3,comment,NaN,Why not both?,64,NaN,NaN,1.627800e+09,13,3,1,...,1.000000,0.076923,1,0,0,0,[],0,0,2021-08-01 06:40:18
4,comment,NaN,I've not talked about this much but as a child...,25,NaN,NaN,1.627817e+09,679,126,0,...,0.746032,0.016200,17,0,0,0,[],0,0,2021-08-01 11:15:58


## Dataset de Publicaciones y Comentarios de r/asktransgender

### Descripción General

Este dataset contiene una colección de **1,252 registros** extraídos de publicaciones (`submissions`) y comentarios (`comments`) del subreddit **r/asktransgender**. El objetivo principal de la recolección fue crear un corpus de texto enriquecido con múltiples características lingüísticas y de sentimiento para su análisis.

Cada registro ha sido procesado para extraer un total de **25 atributos**, incluyendo metadatos de Reddit, métricas textuales, análisis de sentimiento e indicadores de patrones específicos (como la presencia de URLs, menciones o subreddits sugeridos).

### Origen de los Datos

- **Fuente:** Reddit, específicamente el subreddit [r/asktransgender](https://www.reddit.com/r/asktransgender/).
- **Unidad de Observación:** Cada fila del dataset representa una publicación (`submission`) o un comentario (`comment`) individual.
- **Fecha de Recolección:** El contenido fue recolectado en [**POR FAVOR, AÑADE LA FECHA O RANGO DE FECHAS DE RECOLECCIÓN, ej., "marzo de 2024"].
- **Método de Recolección:** Los datos fueron extraídos utilizando la herramienta de código abierto [bulk-downloader-for-reddit](https://github.com/Serene-Arc/bulk-downloader-for-reddit), que genera un archivo JSON por cada elemento de Reddit.
- **Procesamiento:** Posteriormente, se utilizó un script de Python personalizado para leer los archivos JSON, extraer los campos relevantes y aplicar un módulo de extracción de características de texto (ver sección "Ingeniería de Atributos").

### Estructura del Dataset

El archivo `asktransgender_dataset.csv` contiene las siguientes columnas:

| Atributo | Tipo | Descripción |
| :--- | :--- | :--- |
| `type` | Categórico | Tipo de contenido: `submission` (publicación) o `comment` (comentario). |
| `title` | Texto | Título de la publicación. Vacío para los comentarios. |
| `text` | Texto | Cuerpo del texto de la publicación o comentario. |
| `score` | Numérico | Puntuación (upvotes - downvotes) en el momento de la recolección. |
| `upvote_ratio` | Numérico | Proporción de upvotes sobre el total de votos. |
| `num_comments` | Numérico | Número de comentarios en una publicación. Vacío para los comentarios. |
| `created_utc` | Numérico | Fecha y hora de creación en formato timestamp UTC (Unix). |
| `text_length` | Numérico | Longitud total del texto (número de caracteres). |
| `word_count` | Numérico | Número de palabras en el texto. |
| `question_mark_count` | Numérico | Número de signos de interrogación (`?`). |
| `exclamation_mark_count` | Numérico | Número de signos de exclamación (`!`). |
| `sentiment_score` | Numérico | Puntuación de sentimiento (polaridad) de -1 (negativo) a 1 (positivo). |
| `sentiment_subjectivity` | Numérico | Grado de subjetividad del texto de 0 (objetivo) a 1 (subjetivo). |
| `avg_word_length` | Numérico | Longitud promedio de las palabras. |
| `unique_word_ratio` | Numérico | Proporción de palabras únicas sobre el total de palabras. |
| `uppercase_ratio` | Numérico | Proporción de caracteres en mayúscula sobre el total. |
| `punctuation_count` | Numérico | Número total de signos de puntuación. |
| `has_url` | Binario | `1` si el texto contiene una URL, `0` en caso contrario. |
| `has_mention` | Binario | `1` si el texto contiene una mención a un usuario (`/u/` o `@`), `0` en caso contrario. |
| `has_suggested_subreddit` | Binario | `1` si el texto menciona otro subreddit (`r/`), `0` en caso contrario. |
| `suggested_subreddit` | Lista | Lista de subreddits mencionados en el texto (ej., `['asktransgender']`). |
| `has_emoji` | Binario | `1` si el texto original contenía emojis, `0` en caso contrario. |
| `caps_words_count` | Numérico | Número de palabras escritas completamente en mayúsculas (con longitud > 1). |

*Nota: Los campos vacíos en el CSV representan valores faltantes (`NaN`).*

### Atributos

Los atributos numéricos y binarios fueron generados a partir del campo `text` utilizando un script de Python que realiza las siguientes operaciones:

1.  **Detección de Emojis:** Se identifica la presencia de emojis antes de ser procesados.
2.  **Demojización:** Los emojis se convierten a su descripción textual (ej., `😊` se convierte en `:smiling_face_with_smiling_eyes:`) para su análisis.
3.  **Métricas de Texto:** Se calculan longitudes, conteo de palabras y puntuación.
4.  **Análisis de Sentimiento:** Se utiliza la librería `TextBlob` para obtener la polaridad (sentimiento) y la subjetividad del texto.
5.  **Análisis de Caso y Patrones:** Se calculan proporciones de mayúsculas y se buscan patrones específicos como URLs, menciones y referencias a subreddits.
